# Problem Statement : House Prices Prediction using TensorFlow Decision Forests

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this playground competition's dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence.

### Step to Follow

1. Requierd Library
2. Load dataset
3. Understand the data

In [84]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


# 1) Requierd Library

In [85]:
import pandas as pd 
import numpy as np
!pip install tensorflow_decision_forests
from sklearn.model_selection import train_test_split
import tensorflow_decision_forests as tfdf


  Using cached tensorflow_decision_forests-1.8.1.tar.gz (15.2 MB)
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of tensorflow-decision-forests to determine which version is compatible with other requirements. This could take a while.
ERROR: Could not find a version that satisfies the requirement tensorflow~=2.15.0 (from tensorflow-decision-forests) (from versions: 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0, 2.22.0rc0)
ERROR: No matching distribution found for tensorflow~=2.15.0


ModuleNotFoundError: No module named 'tensorflow_decision_forests'

In [ ]:
!pip install ydf

In [ ]:
url = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv'
df = pd.read_csv(url)

Understand dataset size

In [ ]:
shape_of_the_data = df.shape

print('Shpe of the data :',shape_of_the_data)
print('view the top reocrd',df.head())

# Find the missing values
let's understand the missing value why becauss the missing value can make big differnt so we to be carful.

Formula is  sum_of_all_value / Numbser_of_values *100 or 

[X1+X2+X+X4+X5...XN/ N *100](http://)

In [ ]:
def handling_missing_value(data):
    missing = data.isnull().sum()
    missing_per = (missing / len(data)*100).round(2)
    result = pd.DataFrame({'Missing_Count' : missing , 'missing_%' : missing_per })
    return result[result['Missing_Count']>0].sort_values('missing_%',ascending= False)
handling_missing_value(df)

# 3) Understand the data
Check the data type of each if we know that would be better to make the decession 
so we have 
  * object   43
  * int      35
  * float    3

In [ ]:
print(df.dtypes.value_counts())
print(len(df.select_dtypes(include='object').columns.tolist()))

## list the number columns

we have total of 38 numarical columns 

In [ ]:
num_cols = df.select_dtypes(exclude='object').columns.tolist()
print(len(num_cols))
print(num_cols)

# Count the unique values in suspicious columns

In [ ]:
print(df['MSSubClass'].nunique(), df['MoSold'].nunique())

# Prove it with the data (don’t just trust my claim)

In [ ]:
print((df['PoolArea']== 0 ).sum())
print((df['Fireplaces']== 0 ).sum())
print((df['GarageArea']== 0 ).sum())
print((df['TotalBsmtSF']== 0 ).sum())
print((df['GarageCond']== 0 ).sum())



In [ ]:
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'MasVnrType']
df[none_cols] = df[none_cols].fillna('None')

df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())
df['MasVnrArea'] = df['MasVnrArea'].fillna(0)
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['GarageYrBlt'].mean())
handling_missing_value(df)

**All the value's has been fixed now **

In [ ]:
print(df['SalePrice'].describe())
print("Mean:", df['SalePrice'].mean())
print("Median:", df['SalePrice'].median())
print("Skew:", df['SalePrice'].skew())
df['SalePrice'].hist(bins=50)

1. Based on the record we can see that total rows *[1460](http://)*

2. Avg price of the each home *[180921.195890](http://)*

3. what we can understand that from the visual is  most of the house in the range between of *[1.5L](http://)* to *[2L](http://)*.

4. very less we can see the max amount of the price *[7.5L](http://)* also we can see very less min of the home price less then 80k 

In [ ]:
print(df[['Neighborhood', 'SalePrice']].groupby('Neighborhood').mean().sort_values('SalePrice', ascending=False))

# as we can se the salse by Neighborhood or place 
1. the moset expencive place is '[NoRidge       335295.317073](http://)'
2. most least place is MeadowV        98576.470588

In [ ]:
stats = df.groupby('Neighborhood')['SalePrice'].agg(['count', 'mean', 'median']).sort_values('mean')
print(stats)

## Find which number columns move with SalePrice

In [ ]:
corr = df.corr(numeric_only=True)['SalePrice'].sort_values(ascending=False)
print(corr.head(10))
print(corr.tail(5))

# Understanding of the correlation.
1) see the price based on what feture  is increasing the price.

2) Based on [overall quality](http://) the price is increasing we can understand much more that lets see in the ploting.

3) when the [living area](http://) is increasing the price is also increasing.

4) the whole understanding is when the area is growing or if it new house or new feture is added in the home there will be chance to go higher price. 

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(df['GrLivArea'], df['SalePrice'], alpha=0.5)
axes[0].set_title('GrLivArea vs SalePrice')
df.groupby('OverallQual')['SalePrice'].mean().plot(kind='bar', ax=axes[1], title='Average price by OverallQual')
plt.tight_layout()
plt.show()

we can see in the plot how price's are increasing when i have minmal sqf area my cost is less but when i have big living area my price is increasing also in the over all quality too when my 

# What do the prices look like, and does the log fix their shape?

In [ ]:
price = df['SalePrice']
print(price.describe())
print("Mean:", round(price.mean()), "| Median:", round(price.median()))
print("Skew raw:", round(price.skew(), 2))
print("Skew after log:", round(np.log1p(price).skew(), 2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
price.hist(bins=50, ax=axes[0]); axes[0].set_title('SalePrice')
np.log1p(price).hist(bins=50, ax=axes[1]); axes[1].set_title('log(SalePrice)')
plt.show()

looks at the house prices to see their shape. It prints the cheapest price, the most expensive price and the typical range. It compares the mean and the median to see if a few very expensive houses pull the average up. It also measures the skew, which is a number for how lopsided the prices are, and then measures it again after taking the log. The two charts show the same thing: raw prices on the left and log prices on the right.

In [ ]:
y_log = np.log1p(df['SalePrice'])
guess = y_log.mean()
baseline_rmse = np.sqrt(((y_log - guess) ** 2).mean())
print("Baseline RMSE (log scale):", round(baseline_rmse, 3))

In [ ]:

X = df.drop(columns=['Id', 'SalePrice'])
y = np.log1p(df['SalePrice'])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

val_baseline = np.sqrt(((y_val - y_train.mean()) ** 2).mean())
print("Train houses:", len(X_train), "| Validation houses:", len(X_val))
print("Baseline RMSE on validation:", round(val_baseline, 3))

In [ ]:

import ydf

train_df = X_train.copy()
train_df['target'] = y_train

model = ydf.RandomForestLearner(label='target', task=ydf.Task.REGRESSION).train(train_df)

pred = model.predict(X_val)
rmse = np.sqrt(((y_val - pred) ** 2).mean())
print("Model RMSE:", round(rmse, 3), "| Baseline RMSE:", round(val_baseline, 3))

In [ ]:
imp = model.variable_importances()
print(list(imp.keys()))
print(imp["SUM_SCORE"][:10])

errors = pd.DataFrame({'actual': np.expm1(y_val), 'predicted': np.expm1(pred)})
errors['pct_error'] = (errors['predicted'] - errors['actual']) / errors['actual'] * 100
worst = errors.reindex(errors['pct_error'].abs().sort_values(ascending=False).index)
print(worst.head(10).round(0))

In [ ]:
# Which columns did the model use most?
imp = model.variable_importances()
print(list(imp.keys()))
print(imp["SUM_SCORE"][:10])

# Which houses did it get most wrong? (converted back to dollars)
errors = pd.DataFrame({'actual': np.expm1(y_val), 'predicted': np.expm1(pred)})
errors['pct_error'] = (errors['predicted'] - errors['actual']) / errors['actual'] * 100
worst = errors.reindex(errors['pct_error'].abs().sort_values(ascending=False).index)
print(worst.head(10).round(0))


In [ ]:
cols = ['OverallQual', 'GrLivArea', 'Neighborhood', 'SaleCondition', 'YearBuilt', 'OverallCond']
print(df.loc[worst.head(10).index, cols])

In [ ]:
print(df['SaleCondition'].value_counts(normalize=True).round(2))
print(df['Neighborhood'].isin(['IDOTRR', 'OldTown']).mean().round(2))

In [ ]:
gbt = ydf.GradientBoostedTreesLearner(label='target', task=ydf.Task.REGRESSION).train(train_df)

pred_gbt = gbt.predict(X_val)
rmse_gbt = np.sqrt(((y_val - pred_gbt) ** 2).mean())
print("Baseline:", round(val_baseline, 3), "| Random Forest:", round(rmse, 3), "| Gradient Boosted:", round(rmse_gbt, 3))

In [ ]:
from sklearn.model_selection import KFold

full = X.copy()
full['target'] = y

def cv_rmse(learner_class, data, folds=5):
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in kf.split(data):
        tr = data.iloc[train_idx]
        va = data.iloc[val_idx]
        m = learner_class(label='target', task=ydf.Task.REGRESSION).train(tr)
        p = m.predict(va.drop(columns=['target']))
        scores.append(np.sqrt(((va['target'] - p) ** 2).mean()))
    return scores

rf_scores = cv_rmse(ydf.RandomForestLearner, full)
gbt_scores = cv_rmse(ydf.GradientBoostedTreesLearner, full)

print("Random Forest folds:   ", np.round(rf_scores, 3), "| mean:", round(np.mean(rf_scores), 3))
print("Gradient Boosted folds:", np.round(gbt_scores, 3), "| mean:", round(np.mean(gbt_scores), 3))


In [ ]:
X2 = X.copy()
X2['TotalSF'] = X2['TotalBsmtSF'] + X2['1stFlrSF'] + X2['2ndFlrSF']
X2['HouseAge'] = X2['YrSold'] - X2['YearBuilt']
X2['TotalBath'] = X2['FullBath'] + 0.5 * X2['HalfBath'] + X2['BsmtFullBath'] + 0.5 * X2['BsmtHalfBath']

full2 = X2.copy()
full2['target'] = y

rf_scores2 = cv_rmse(ydf.RandomForestLearner, full2)
print("Before:", round(np.mean(rf_scores), 3), "| After new features:", round(np.mean(rf_scores2), 3))
print("Folds after:", np.round(rf_scores2, 3))

In [ ]:
test = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')   # use the os.walk snippet to find your path

none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'MasVnrType']

def prepare(df, lot_median, elec_mode):
    df = df.copy()
    df['MSSubClass'] = df['MSSubClass'].astype(str)
    df['MoSold'] = df['MoSold'].astype(str)
    df[none_cols] = df[none_cols].fillna('None')
    df['LotFrontage'] = df['LotFrontage'].fillna(lot_median)
    df['MasVnrArea'] = df['MasVnrArea'].fillna(0)
    df['Electrical'] = df['Electrical'].fillna(elec_mode)
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']
    return df

lot_median = df['LotFrontage'].median()
elec_mode = df['Electrical'].mode()[0]

test_ready = prepare(test, lot_median, elec_mode)
X_test = test_ready[X2.columns]
assert list(X_test.columns) == list(X2.columns)

final = ydf.RandomForestLearner(label='target', task=ydf.Task.REGRESSION).train(full2)
test_pred = np.expm1(final.predict(X_test))

submission = pd.DataFrame({'Id': test['Id'], 'SalePrice': test_pred})
submission.to_csv('submission.csv', index=False)
print(submission.shape)
print(submission['SalePrice'].describe().round(0))

In [ ]:
print(submission['SalePrice'].isnull().sum())

In [ ]:
X['MSSubClass'] = X['MSSubClass'].astype(str)
X['MoSold'] = X['MoSold'].astype(str)